# 凸包檢測(幾何形狀分類)_修正版.py
**電腦視覺與深度學習 › Day14 OpenCV 電腦視覺**  ·  ✅ 修正版

凸包檢測原本有兩個問題:圓心比對同樣有 uint16 溢位,而且只看頂點數,把長方形也叫成 Square。

> 🎧 有逐段旁白、逐句拆解的版本在 [pyl Academy](https://friencdking0000.github.io/pyl-academy/#/learn/PY-CVAI/0/4)。
>
> **怎麼用**:由上往下,每格按左邊 ▶ 執行;或選單「執行階段 › 全部執行」。
>
> 📷 Colab 沒有攝影機、也開不了視窗:下面會**改用範例圖片**代替鏡頭,影像直接顯示在格子底下(只顯示前幾張)。

In [ ]:
#@title 0. 準備環境(下載課程範例檔、安裝套件)
!git clone -q --depth 1 https://github.com/friencdking0000/2026AI-STUST.git /content/2026AI-STUST 2>/dev/null; echo 課程檔案已就緒
%cd "/content/2026AI-STUST/Week1/DAY7"
import sys, os; sys.argv = sys.argv[:1]  # 讓 argparse / sys.argv 在 notebook 裡也正常
__file__ = os.path.join(os.getcwd(), '凸包檢測(幾何形狀分類).py')  # notebook 沒有 __file__,程式靠它找同資料夾的檔案

In [ ]:
#@title 1. Colab 沒有攝影機和視窗:改用範例圖片、影像直接顯示在下面
import cv2, numpy as np, urllib.request, os
from google.colab.patches import cv2_imshow
def _pyl_sample():
    img = np.full((480, 640, 3), 255, np.uint8)
    cv2.circle(img, (130, 140), 70, (0, 0, 255), -1)
    cv2.circle(img, (520, 360), 55, (255, 0, 0), -1)
    cv2.rectangle(img, (260, 60), (400, 200), (0, 160, 0), -1)
    cv2.fillPoly(img, [np.array([[90, 420], [230, 420], [160, 300]])], (0, 200, 255))
    cv2.fillPoly(img, [np.array([[300, 300], [430, 280], [450, 420], [320, 440]])], (200, 0, 200))
    return img
class _PylCam:
    def __init__(self, *a, **k): self.n = 0; self.img = _pyl_sample()
    def isOpened(self): return True
    def read(self):
        self.n += 1
        return (True, self.img.copy()) if self.n <= 30 else (False, None)
    def set(self, *a): return True
    def get(self, *a): return 0
    def release(self): pass
_pyl_VC = cv2.VideoCapture
cv2.VideoCapture = lambda src=0, *a, **k: _PylCam() if isinstance(src, int) else _pyl_VC(src, *a, **k)
_pyl_shown = [0]
def _pyl_imshow(name, img):
    _pyl_shown[0] += 1
    if _pyl_shown[0] <= 6:
        print('▼', name); cv2_imshow(img)
cv2.imshow = _pyl_imshow
cv2.waitKey = lambda *a: ord('q') if _pyl_shown[0] >= 6 else -1
cv2.destroyAllWindows = lambda *a: None
cv2.namedWindow = lambda *a, **k: None
cv2.createTrackbar = lambda *a, **k: None
cv2.getTrackbarPos = lambda *a, **k: 0
cv2.setMouseCallback = lambda *a, **k: None
cv2.resizeWindow = cv2.moveWindow = cv2.setWindowProperty = cv2.setWindowTitle = lambda *a, **k: None
_pyl_imread = cv2.imread
def _pyl_imread2(path, *a, **k):
    img = _pyl_imread(path, *a, **k)
    if img is None and not os.path.exists(str(path)):
        print('(找不到', path, ':課程 repo 沒附這張圖,改用範例圖片)')
        img = _pyl_sample()
    return img
cv2.imread = _pyl_imread2
print('已改用範例圖片代替攝影機')

### 修正版說明
開頭註解列出兩個修正:圓心先轉 int,以及用寬高比分出正方形和長方形。

In [ ]:
# 修正版:對照原檔 凸包檢測(幾何形狀分類).py(原檔保留當「錯誤示範」)
# 修正 1:霍夫圓心是 np.uint16,直接和 int 相減會溢位繞回,可能把遠處的圓誤配成同一個;改成先轉 int。
# 修正 2:4 個頂點一律標成 Square,長方形也被叫成正方形;加上寬高比 0.85~1.15 判斷,
#         其餘標成 Rectangle(規則與 視角切換與形狀偵測.py 相同)。
import cv2
import numpy as np

# 1. 建立 VideoCapture 物件
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("錯誤：無法開啟視訊鏡頭。")
    exit()

print("鏡頭已開啟！幾何形狀檢測中... 按下 'q' 鍵可結束程式。")

# 定義目標面積區間
min_area = 500   # 稍微調大一點點，避免雜訊
max_area = 50000 

### 四邊形再看寬高比
錯在哪:原本只要 4 個頂點就標 Square。為什麼:頂點數分不出正方形和長方形。怎麼修:用 boundingRect 算寬高比,0.85 到 1.15 之間才是正方形,其他標 Rectangle,和視角切換那支的規則一樣。怎麼驗證:假畫面放一個 180 乘 80 的長方形,原檔標成 Square,修正版標成 Rectangle;100 乘 100 的正方形兩版都是 Square。

### 圓心先轉 int
錯在哪、為什麼:和霍夫圓檢測一樣,uint16 相減會繞回。怎麼修:先用 int 轉換。怎麼驗證:把假的霍夫圓心放在真正圓心右邊 256 像素,原檔仍然標出 Circle,修正版不會;霍夫圓心放在正確位置時,修正版照樣標出 Circle。

In [ ]:
while True:
    ret, frame = cap.read()
    if not ret:
        print("錯誤：無法接收影像畫面。")
        break

    img_copy = frame.copy()
    
    # 影像預處理
    gray_img = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray_img, (7, 7), 1.5)
    
    # A. 邊緣與輪廓檢測
    edges = cv2.Canny(blurred, 30, 150)
    contours, hierarchy = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # B. 霍夫圓檢測 (預備給圓形比對使用)
    circles = cv2.HoughCircles(blurred, cv2.HOUGH_GRADIENT, dp=1, minDist=50,
                               param1=100, param2=35, minRadius=10, maxRadius=200)
    if circles is not None:
        circles = np.uint16(np.around(circles))

    for cnt in contours:
        # 1. 面積條件過濾
        area = cv2.contourArea(cnt)
        if min_area <= area <= max_area:
            
            # 2. 凸包檢測 (Convex Hull)
            hull = cv2.convexHull(cnt)
            
            # 3. 多邊形逼近 (Polygon Approximation)
            # 這裡的 0.04 是逼近精準度，可根據視訊鏡頭品質微調 (通常在 0.02 ~ 0.05 之間)
            epsilon = 0.04 * cv2.arcLength(hull, True)
            approx = cv2.approxPolyDP(hull, epsilon, True)
            
            # 計算形狀的中心點
            M = cv2.moments(cnt)
            if M["m00"] != 0:
                cX = int(M["m10"] / M["m00"])
                cY = int(M["m01"] / M["m00"])
            else:
                continue

            shape_label = "Unknown"
            color = (255, 255, 255) # 預設白色
            vertices_count = len(approx)

            # 4. 依據幾何特徵進行分類
            
            # 【情況 A：三角形】
            if vertices_count == 3:
                shape_label = "Triangle"
                color = (0, 255, 255)  # 黃色
                
            # 【情況 B：正方形/四邊形】
            elif vertices_count == 4:
                # 【修正 2】用寬高比分正方形和長方形
                x, y, w, h = cv2.boundingRect(approx)
                ratio = w / float(h) if h > 0 else 0
                shape_label = "Square" if 0.85 <= ratio <= 1.15 else "Rectangle"
                color = (0, 255, 0)    # 綠色
                
            # 【情況 C：圓形辨識】(頂點數較多時，結合霍夫圓比對)
            elif vertices_count > 4 and circles is not None:
                for circle in circles[0, :]:
                    # 【修正 1】uint16 先轉 int,相減才不會繞回
                    h_x, h_y = int(circle[0]), int(circle[1])
                    distance = np.sqrt((cX - h_x)**2 + (cY - h_y)**2)
                    
                    if distance < 20: # 圓心距離在合理範圍內
                        shape_label = "Circle"
                        color = (0, 0, 255)  # 紅色
                        break
            
            # 5. 如果成功分類，就繪製出來
            if shape_label != "Unknown":
                # 繪製凸包輪廓
                cv2.drawContours(img_copy, [hull], -1, color, 2)
                
                # 繪製中心點
                cv2.circle(img_copy, (cX, cY), 4, color, -1)
                
                # 標註圖形名稱與面積
                text = f"{shape_label} (A:{int(area)})"
                cv2.putText(img_copy, text, (cX - 40, cY - 10), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    # 3. 顯示結果
    cv2.imshow('debug', edges)
    cv2.imshow('Webcam Shape Detection', img_copy)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

## 小結
修正後,三角形、正方形、長方形、圓都分得清楚,遠處的霍夫圓也不會被誤配。

👉 回到 [pyl Academy 這堂課](https://friencdking0000.github.io/pyl-academy/#/learn/PY-CVAI/0/4) 看逐句拆解、做練習。